# CLIP4CAD-HUS Training

Train the **Hierarchical Unified Space (HUS)** architecture for CAD multimodal learning.

## Key Differences from GFA

| Aspect | GFA | HUS |
|--------|-----|-----|
| Architecture | Text parsing → Grounding matrices | Global queries → Detail queries (conditioned) |
| Levels | Single (grounded features) | Two (Global + Detail) |
| Queries | K=12 text-parsed slots | G=8 global + M=24 detail, modality-adapted |
| Loss terms | 8 terms | 3 terms (simplified) |
| Inference | Requires self_ground_queries | Direct (queries work without text) |

## Training Stages

- **Stage 1 (epochs 1-15):** Hierarchy establishment, standard InfoNCE
- **Stage 2 (epochs 16-35):** Hard negative mining at detail level, increased detail weight

## Features

- Simplified 3-term loss (unified, global, detail)
- Gate value monitoring for interpretability
- Modality-aware queries with Tanh-bounded adapters
- Global-conditioned detail attention

## Usage

1. Run Cells 1-3 to load data (~5 min, only once)
2. Run Cell 4 to train the full model
3. Run Cell 5+ for evaluation

---
## Cell 1: Imports & Setup

In [2]:
import sys
from pathlib import Path

# Add project root to path
sys.path.insert(0, 'D:/Defect_Det/MMCAD/MMCAD')

import gc
import torch
from omegaconf import OmegaConf

from clip4cad.data.gfa_dataset import GFAMappedDataset
from clip4cad.models.clip4cad_hus import CLIP4CAD_HUS_v2
from clip4cad.training.hus_trainer import HUSTrainer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\n" + "="*60)
print("Run Cell 2 to configure paths, then Cell 3 to load data.")
print("="*60)

Device: cuda
PyTorch version: 2.5.1+cu121
GPU: NVIDIA GeForce RTX 4090
Memory: 25.8 GB

Run Cell 2 to configure paths, then Cell 3 to load data.


---
## Cell 2: Configuration

In [10]:
# =============================================================================
# PATHS - Update these for your setup
# =============================================================================
PATHS = {
    "data_root": "d:/Defect_Det/MMCAD/data",
    "pc_file": "c:/Users/User/Desktop/pc_embeddings_full.h5",
    "brep_file": "c:/Users/User/Desktop/brep_features.h5",
    "text_dir": "c:/Users/User/Desktop/text_splits/",  # Pre-split text embeddings
    "config_path": "d:/Defect_Det/MMCAD/MMCAD/configs/model/clip4cad_hus.yaml",
    "output_dir": "outputs/hus",
}

# =============================================================================
# TRAINING CONFIG - Override config values if needed
# =============================================================================
TRAIN_CONFIG = {
    "batch_size": 512,
    "num_workers": 0,  # CRITICAL: 0 because data is in RAM
    "num_epochs_stage1": 15,
    "num_epochs_stage2": 20,
    # CHANGED: Higher learning rate (was 3e-5)
    "learning_rate": 3e-4,

    # CHANGED: Slightly longer warmup for stability with high LR
    "warmup_epochs": 2,

    "save_every": 5,
    "validate_every": 5,

    # CHANGED: Unified-dominant loss weights
    "lambda_unified": 1.0,      # PRIMARY (was 1.0, keep)
    "lambda_global": 0.2,       # Mild regularizer (was 0.3)
    "lambda_detail": 0.2,       # Mild regularizer (was 0.2, keep)
    "label_smoothing": 0.05,    # NEW: Light smoothing

    # Stage 2 weights
    "lambda_unified_stage2": 1.0,   # Still primary
    "lambda_global_stage2": 0.1,    # Reduce further (was 0.5)
    "lambda_detail_stage2": 0.5,    # For hard negatives (was 1.0)
    }

print("Configuration:")
print(f"  Batch size: {TRAIN_CONFIG['batch_size']}")
print(f"  Learning rate: {TRAIN_CONFIG['learning_rate']}")
print(f"  Stage 1: {TRAIN_CONFIG['num_epochs_stage1']} epochs")
print(f"  Stage 2: {TRAIN_CONFIG['num_epochs_stage2']} epochs")
print(f"  Total: {TRAIN_CONFIG['num_epochs_stage1'] + TRAIN_CONFIG['num_epochs_stage2']} epochs")
print(f"\nLoss weights:")
print(f"  lambda_unified: {TRAIN_CONFIG['lambda_unified']}")
print(f"  lambda_global: {TRAIN_CONFIG['lambda_global']}")
print(f"  lambda_detail: {TRAIN_CONFIG['lambda_detail']} (stage 1)")
print(f"  lambda_detail: {TRAIN_CONFIG['lambda_detail_stage2']} (stage 2)")
print(f"\nOutput: {PATHS['output_dir']}")

Configuration:
  Batch size: 512
  Learning rate: 0.0003
  Stage 1: 15 epochs
  Stage 2: 20 epochs
  Total: 35 epochs

Loss weights:
  lambda_unified: 1.0
  lambda_global: 0.2
  lambda_detail: 0.2 (stage 1)
  lambda_detail: 0.5 (stage 2)

Output: outputs/hus


---
## Cell 3: Load Datasets (RUN ONCE)

**This loads data into RAM. Takes a few minutes but only needs to run ONCE.**

In [4]:
print("Loading datasets...")
print("=" * 60)

# Train dataset - LOAD TO RAM for faster training
print("\n[1/2] Loading TRAIN dataset to RAM...")
train_dataset = GFAMappedDataset(
    data_root=PATHS["data_root"],
    split="train",
    pc_file=PATHS["pc_file"],
    text_file=PATHS["text_dir"],  # Will resolve to train_text_embeddings.h5
    brep_file=PATHS["brep_file"],
    num_rotations=1,
    load_to_memory=True,
    use_live_text=False,
)
print(f"Train: {len(train_dataset):,} samples in RAM")

# Val dataset - ON DISK (saves RAM)
print("\n[2/2] Loading VAL dataset (on disk)...")
val_dataset = GFAMappedDataset(
    data_root=PATHS["data_root"],
    split="val",
    pc_file=PATHS["pc_file"],
    text_file=PATHS["text_dir"],  # Will resolve to val_text_embeddings.h5
    brep_file=PATHS["brep_file"],
    num_rotations=1,
    load_to_memory=False,
    use_live_text=False,
)
print(f"Val: {len(val_dataset):,} samples on disk")

print("\n" + "=" * 60)
print("DATASETS READY!")
print("Do NOT re-run this cell unless you restart the kernel!")
print("=" * 60)

Loading datasets...

[1/2] Loading TRAIN dataset to RAM...
  Loading train data to memory (B-Rep + PC + Text)...
    Loading B-Rep (3GB)...
    Loading PC (50GB)...
    Loading Text from: c:\Users\User\Desktop\text_splits\train_text_embeddings.h5
    ✓ Text loaded: 174.6GB in 343.1s
    ⚠️  Pre-split has 111000 samples, dataset expected 133105
    Using first 111000 samples to match pre-split file
  ✓ Loaded 111000 samples: 203.6GB in RAM (B-Rep + PC + Text)
GFAMappedDataset: train with 111000 samples (in memory)
Train: 111,000 samples in RAM

[2/2] Loading VAL dataset (on disk)...
GFAMappedDataset: val with 16638 samples
Val: 16,638 samples on disk

DATASETS READY!
Do NOT re-run this cell unless you restart the kernel!


---
## Cell 4: Train CLIP4CAD-HUS

Full 2-stage training:
- Stage 1: Hierarchy establishment (15 epochs)
- Stage 2: Hard negative mining at detail level (20 epochs)

In [11]:
print("\n" + "=" * 70)
print("CLIP4CAD-HUS Training")
print("=" * 70)

# Load config
config = OmegaConf.load(PATHS["config_path"])

# Override with TRAIN_CONFIG
for key, value in TRAIN_CONFIG.items():
    if hasattr(config, 'training') and key in config.training:
        config.training[key] = value

# Output directory
output_dir = Path(PATHS["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)

# Save config
OmegaConf.save(config, output_dir / "config.yaml")

# Create model
print("\nCreating CLIP4CAD-HUS model...")
model = CLIP4CAD_HUS_v2(config)
param_counts = model.count_parameters()
print(f"  Total parameters: {param_counts['total']:,}")
print(f"  Trainable parameters: {param_counts['trainable']:,}")
print(f"  Query bank: {param_counts.get('query_bank', 0):,}")
print(f"  Global attention: {param_counts.get('global_attn', 0):,}")
print(f"  Detail attention: {param_counts.get('detail_attn', 0):,}")
print(f"  Fusion: {param_counts.get('fusion', 0):,}")

# Create trainer
print("\nInitializing HUS trainer...")
trainer = HUSTrainer(
    model=model,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    config=OmegaConf.to_container(config.training),
    output_dir=str(output_dir),
    device=str(device),
)

# Train
print("\nStarting training...")
print(f"  Stage 1: {TRAIN_CONFIG['num_epochs_stage1']} epochs (hierarchy establishment)")
print(f"  Stage 2: {TRAIN_CONFIG['num_epochs_stage2']} epochs (hard negative mining)")
trainer.train()

print("\n" + "=" * 70)
print("TRAINING COMPLETE!")
print(f"Checkpoints saved to: {output_dir}")
print("=" * 70)

# Cleanup for next training
del trainer
gc.collect()
torch.cuda.empty_cache()


CLIP4CAD-HUS Training

Creating CLIP4CAD-HUS model...
  Total parameters: 3,807,109
  Trainable parameters: 3,807,109
  Query bank: 206,336
  Global attention: 789,760
  Detail attention: 926,976
  Fusion: 263,682

Initializing HUS trainer...

Starting training...
  Stage 1: 15 epochs (hierarchy establishment)
  Stage 2: 20 epochs (hard negative mining)
CLIP4CAD-HUS Training
Total epochs: 35
Stage 1: 15 epochs (hierarchy)
Stage 2: 20 epochs (fine-grained)
Batch size: 512
Learning rate: 0.0003
Total parameters: 3,807,109
Trainable parameters: 3,807,109
Loss terms: 3 (unified, global, detail)
Output directory: outputs\hus



Epoch 1 (Stage 1):   0%|          | 1/216 [00:01<04:38,  1.29s/it]


  [WARNING] NaN/Inf gradients at batch 0, skipping update


Epoch 1 (Stage 1):   1%|          | 2/216 [00:02<04:09,  1.17s/it]


  [WARNING] NaN/Inf gradients at batch 1, skipping update


...


  [Gates] brep=0.37/0.63 pc=0.46/0.54 text=0.35/0.65


Epoch 35 (Stage 2): 100%|██████████| 216/216 [03:29<00:00,  1.03it/s, loss=4.671, U=2.733, G=3.466, D=3.181, lr=1.0e-06]



  [NaN Summary] 4/216 batches had NaN (1.9%)


Validation: 100%|██████████| 33/33 [02:15<00:00,  4.09s/it]



Epoch 35 Summary:
  Train Loss: 4.3867
    Unified: 2.5542
    Global:  3.3870
    Detail:  2.9875
  Gate Values (global/detail):
    B-Rep: 0.363/0.637
    PC:    0.450/0.550
    Text:  0.346/0.654
  Val Loss: 6.3410
Final model saved: outputs\hus\clip4cad_hus_final.pt

Training Complete!

TRAINING COMPLETE!
Checkpoints saved to: outputs\hus


---
## Cell 5: Resume Training (Optional)

Resume from a checkpoint if training was interrupted.

In [ ]:
# RESUME TRAINING (uncomment to use)

# resume_checkpoint = "outputs/hus/checkpoint_epoch15.pt"  # Change this

# config = OmegaConf.load(PATHS["config_path"])
# model = CLIP4CAD_HUS_v2(config)

# trainer = HUSTrainer(
#     model=model,
#     train_dataset=train_dataset,
#     val_dataset=val_dataset,
#     config=OmegaConf.to_container(config.training),
#     output_dir=PATHS["output_dir"],
#     device=str(device),
# )

# trainer.resume(resume_checkpoint)
# trainer.train()

print("Uncomment the code above to resume training from a checkpoint.")

---
---
# EVALUATION

---

## Evaluate Checkpoint

In [12]:
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from clip4cad.data.gfa_dataset import gfa_collate_fn
import json

def evaluate_hus_checkpoint(checkpoint_path: str, dataset, config_path: str = None):
    """
    Evaluate a HUS checkpoint on retrieval tasks.
    
    Args:
        checkpoint_path: Path to checkpoint
        dataset: Evaluation dataset
        config_path: Path to config (uses default if None)
    
    Returns:
        dict: Evaluation results
    """
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        print(f"Checkpoint not found: {checkpoint_path}")
        return None
    
    print(f"\nEvaluating: {checkpoint_path.name}")
    
    # Load checkpoint
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    
    # Load config
    if config_path is None:
        config_path = PATHS["config_path"]
    config = OmegaConf.load(config_path)
    
    # Create model and load weights
    model = CLIP4CAD_HUS_v2(config)
    model.load_state_dict(ckpt["model_state_dict"])
    model = model.to(device).eval()
    
    # Info
    epoch = ckpt.get("epoch", "?")
    stage = ckpt.get("stage", "?")
    print(f"  Epoch: {epoch}, Stage: {stage}")
    
    # Create loader
    loader = DataLoader(
        dataset, batch_size=64, shuffle=False,
        num_workers=0, pin_memory=True, collate_fn=gfa_collate_fn
    )
    
    # Generate embeddings
    text_emb, brep_emb, pc_emb = [], [], []
    text_global, brep_global, pc_global = [], [], []
    text_detail, brep_detail, pc_detail = [], [], []
    gate_brep, gate_pc, gate_text = [], [], []
    ids = []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="  Embeddings", leave=False):
            ids.extend(batch["sample_id"])
            outputs = model(batch)
            
            # Unified embeddings (primary)
            text_emb.append(F.normalize(outputs["z_text"].float(), p=2, dim=-1).cpu())
            brep_emb.append(F.normalize(outputs["z_brep"].float(), p=2, dim=-1).cpu())
            pc_emb.append(F.normalize(outputs["z_pc"].float(), p=2, dim=-1).cpu())
            
            # Global embeddings
            text_global.append(F.normalize(outputs["z_text_global"].float(), p=2, dim=-1).cpu())
            brep_global.append(F.normalize(outputs["z_brep_global"].float(), p=2, dim=-1).cpu())
            pc_global.append(F.normalize(outputs["z_pc_global"].float(), p=2, dim=-1).cpu())
            
            # Detail embeddings
            text_detail.append(F.normalize(outputs["z_text_detail"].float(), p=2, dim=-1).cpu())
            brep_detail.append(F.normalize(outputs["z_brep_detail"].float(), p=2, dim=-1).cpu())
            pc_detail.append(F.normalize(outputs["z_pc_detail"].float(), p=2, dim=-1).cpu())
            
            # Gate values
            gate_brep.append(outputs["gate_brep"].cpu())
            gate_pc.append(outputs["gate_pc"].cpu())
            gate_text.append(outputs["gate_text"].cpu())
    
    # Concatenate
    text_emb = torch.cat(text_emb)
    brep_emb = torch.cat(brep_emb)
    pc_emb = torch.cat(pc_emb)
    gate_brep = torch.cat(gate_brep)
    gate_pc = torch.cat(gate_pc)
    gate_text = torch.cat(gate_text)
    
    # Compute retrieval metrics
    k_values = [1, 5, 10]
    results = {"epoch": epoch, "stage": stage}
    
    tasks = [
        ("Text->BRep", text_emb, brep_emb),
        ("Text->PC", text_emb, pc_emb),
        ("PC->BRep", pc_emb, brep_emb),
        ("BRep->PC", brep_emb, pc_emb),
    ]
    
    for task_name, query_emb, gallery_emb in tasks:
        sim = torch.mm(query_emb, gallery_emb.T)
        _, rankings = torch.topk(sim, k=max(k_values), dim=1)
        
        n = len(ids)
        task_results = {}
        
        for k in k_values:
            hits = 0
            ap_sum = 0.0
            
            for i in range(n):
                qid = str(ids[i])
                for rank, idx in enumerate(rankings[i, :k]):
                    if str(ids[idx]) == qid:
                        hits += 1
                        ap_sum += 1.0 / (rank + 1)
                        break
            
            task_results[f"R@{k}"] = hits / n
            task_results[f"mAP@{k}"] = ap_sum / n
        
        results[task_name] = task_results
    
    # Gate statistics
    results["gate_stats"] = {
        "brep_mean": float(gate_brep.mean()),
        "brep_std": float(gate_brep.std()),
        "pc_mean": float(gate_pc.mean()),
        "pc_std": float(gate_pc.std()),
        "text_mean": float(gate_text.mean()),
        "text_std": float(gate_text.std()),
    }
    
    # Cleanup
    del model
    gc.collect()
    torch.cuda.empty_cache()
    
    return results

print("Evaluation function defined.")
print("Usage: results = evaluate_hus_checkpoint('outputs/hus/checkpoint_epoch35.pt', val_dataset)")

Evaluation function defined.
Usage: results = evaluate_hus_checkpoint('outputs/hus/checkpoint_epoch35.pt', val_dataset)


## Evaluate All Checkpoints

In [13]:
# =============================================================================
# EVALUATE MULTIPLE CHECKPOINTS
# =============================================================================

# Which checkpoints to evaluate
EVAL_CHECKPOINTS = [15, 20, 25, 30, 35, "best"]  # Customize this list

output_dir = Path(PATHS["output_dir"])

# Find checkpoints
checkpoints = []
for spec in EVAL_CHECKPOINTS:
    if spec == "best":
        ckpt = output_dir / "checkpoint_best.pt"
    else:
        ckpt = output_dir / f"checkpoint_epoch{spec}.pt"
    if ckpt.exists():
        checkpoints.append(ckpt)
    else:
        print(f"Warning: {ckpt.name} not found")

print("=" * 80)
print(f"CLIP4CAD-HUS EVALUATION")
print(f"Found {len(checkpoints)} checkpoints to evaluate")
print("=" * 80)

all_results = []
for ckpt_path in checkpoints:
    results = evaluate_hus_checkpoint(ckpt_path, val_dataset)
    if results:
        all_results.append(results)
        # Print summary
        tb = results.get("Text->BRep", {})
        print(f"  Text->BRep: R@1={tb.get('R@1', 0):.4f}, R@5={tb.get('R@5', 0):.4f}, R@10={tb.get('R@10', 0):.4f}")

# Summary table
print("\n" + "=" * 100)
print(f"{'Epoch':<8} {'Stage':<6} | {'Text->BRep':^30} | {'Text->PC':^30}")
print(f"{'':8} {'':6} | {'R@1':>10} {'R@5':>10} {'R@10':>10} | {'R@1':>10} {'R@5':>10} {'R@10':>10}")
print("-" * 100)

for res in all_results:
    epoch = res["epoch"]
    stage = res["stage"]
    tb = res.get("Text->BRep", {})
    tp = res.get("Text->PC", {})
    
    print(f"{epoch:<8} {stage:<6} | "
          f"{tb.get('R@1', 0):>10.4f} {tb.get('R@5', 0):>10.4f} {tb.get('R@10', 0):>10.4f} | "
          f"{tp.get('R@1', 0):>10.4f} {tp.get('R@5', 0):>10.4f} {tp.get('R@10', 0):>10.4f}")

print("=" * 100)

# Gate statistics
print("\nGate Statistics (shows global vs detail importance):")
print(f"{'Epoch':<8} | {'BRep Gate':^20} | {'PC Gate':^20} | {'Text Gate':^20}")
print(f"{'':8} | {'mean':>10} {'std':>10} | {'mean':>10} {'std':>10} | {'mean':>10} {'std':>10}")
print("-" * 80)

for res in all_results:
    epoch = res["epoch"]
    gs = res.get("gate_stats", {})
    print(f"{epoch:<8} | "
          f"{gs.get('brep_mean', 0):>10.4f} {gs.get('brep_std', 0):>10.4f} | "
          f"{gs.get('pc_mean', 0):>10.4f} {gs.get('pc_std', 0):>10.4f} | "
          f"{gs.get('text_mean', 0):>10.4f} {gs.get('text_std', 0):>10.4f}")

print("=" * 80)
print("Note: Gate=0.5 means equal global/detail, >0.5 means more global, <0.5 means more detail")

# Save results
results_path = output_dir / "evaluation_results.json"
with open(results_path, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nResults saved to: {results_path}")

CLIP4CAD-HUS EVALUATION
Found 6 checkpoints to evaluate

Evaluating: checkpoint_epoch15.pt
  Epoch: 15, Stage: 1


  Embeddings:   0%|          | 0/260 [00:00<?, ?it/s]

  Text->BRep: R@1=0.0180, R@5=0.0636, R@10=0.1017

Evaluating: checkpoint_epoch20.pt
  Epoch: 20, Stage: 2


  Embeddings:   0%|          | 0/260 [00:00<?, ?it/s]

  Text->BRep: R@1=0.0205, R@5=0.0667, R@10=0.1086

Evaluating: checkpoint_epoch25.pt
  Epoch: 25, Stage: 2


...

  Text->BRep: R@1=0.0225, R@5=0.0734, R@10=0.1155

Evaluating: checkpoint_epoch35.pt
  Epoch: 35, Stage: 2


  Embeddings:   0%|          | 0/260 [00:00<?, ?it/s]

  Text->BRep: R@1=0.0218, R@5=0.0719, R@10=0.1125

Evaluating: checkpoint_best.pt
  Epoch: 10, Stage: 1


  Embeddings:   0%|          | 0/260 [00:00<?, ?it/s]

  Text->BRep: R@1=0.0206, R@5=0.0691, R@10=0.1152

Epoch    Stage  |           Text->BRep           |            Text->PC           
                |        R@1        R@5       R@10 |        R@1        R@5       R@10
----------------------------------------------------------------------------------------------------
15       1      |     0.0180     0.0636     0.1017 |     0.1154     0.2967     0.4042
20       2      |     0.0205     0.0667     0.1086 |     0.1259     0.3114     0.4193
25       2      |     0.0212     0.0730     0.1145 |     0.1292     0.3222     0.4252
30       2      |     0.0225     0.0734     0.1155 |     0.1275     0.3240     0.4296
35       2      |     0.0218     0.0719     0.1125 |     0.1281     0.3220     0.4281
10       1      |     0.0206     0.0691     0.1152 |     0.1010     0.2705     0.3727

Gate Statistics (shows global vs detail importance):
Epoch    |      BRep Gate       |       PC Gate        |      Text Gate      
         |       mean        std

---
## Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt
import json

output_dir = Path(PATHS["output_dir"])
history_path = output_dir / "training_history.json"

if history_path.exists():
    with open(history_path) as f:
        history = json.load(f)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Total loss
    ax = axes[0, 0]
    if "train_loss" in history:
        ax.plot(history["train_loss"], label="Train", linewidth=2)
    if "val_loss" in history:
        ax.plot(history["val_loss"], label="Val", linewidth=2)
    ax.axvline(x=TRAIN_CONFIG["num_epochs_stage1"]-1, color="gray", linestyle="--", alpha=0.5, label="Stage 2")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Total Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Component losses
    ax = axes[0, 1]
    for key in ["train_unified", "train_global", "train_detail"]:
        if key in history:
            label = key.replace("train_", "").title()
            ax.plot(history[key], label=label, linewidth=2)
    ax.axvline(x=TRAIN_CONFIG["num_epochs_stage1"]-1, color="gray", linestyle="--", alpha=0.5)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Component Losses")
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Gate values over training
    ax = axes[1, 0]
    for key in ["gate_brep", "gate_pc", "gate_text"]:
        if key in history:
            label = key.replace("gate_", "").upper()
            ax.plot(history[key], label=label, linewidth=2)
    ax.axvline(x=TRAIN_CONFIG["num_epochs_stage1"]-1, color="gray", linestyle="--", alpha=0.5)
    ax.axhline(y=0.5, color="black", linestyle=":", alpha=0.3, label="Balanced")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Gate Value")
    ax.set_title("Gate Values (Global vs Detail Balance)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1)
    
    # Temperature evolution
    ax = axes[1, 1]
    for key in ["tau_unified", "tau_global", "tau_detail"]:
        if key in history:
            label = key.replace("tau_", "").title()
            ax.plot(history[key], label=label, linewidth=2)
    ax.axvline(x=TRAIN_CONFIG["num_epochs_stage1"]-1, color="gray", linestyle="--", alpha=0.5)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Temperature")
    ax.set_title("Learnable Temperatures")
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_dir / "training_curves.png", dpi=150)
    plt.show()
    print(f"Saved: {output_dir / 'training_curves.png'}")
else:
    print(f"Training history not found: {history_path}")
    print("Train the model first!")

---
## Memory Check

In [ ]:
import psutil

gc.collect()
torch.cuda.empty_cache()

process = psutil.Process()
ram_gb = process.memory_info().rss / 1e9
available_ram = psutil.virtual_memory().available / 1e9

print(f"Process RAM: {ram_gb:.1f} GB")
print(f"Available RAM: {available_ram:.1f} GB")

if torch.cuda.is_available():
    gpu_alloc = torch.cuda.memory_allocated() / 1e9
    gpu_reserved = torch.cuda.memory_reserved() / 1e9
    print(f"GPU Allocated: {gpu_alloc:.1f} GB")
    print(f"GPU Reserved: {gpu_reserved:.1f} GB")